## CS310 Natural Language Processing
## Lab 4: Transformer Basics

In this lab, we will implement the multihead attention with future masking for causal language modeling. 

Install the `tiktoken` library by running:
```bash
pip install tiktoken
```

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

## T1. Implement Self-Attention for a Single Head

First, prepare the input data in shape $N\times d$

*Hint*: 
- Use `torch.randn` to generate a torch tensor in the correct shape.

In [ ]:
N = 3
d = 512
torch.manual_seed(0)

### START YOUR CODE ###
X = None
### END YOUR CODE ###


# Test 
assert isinstance(X, torch.Tensor)
print('X.size():', X.size())
print('X[:,0]:', X[:,0].data.numpy())

# You should expect to see the following results:
# X.shape: (3, 512)
# X[:,0]: [-1.1258398  -0.54607284 -1.0840825 ]

Then, initialize weight matrices $W^Q$, $W^K$, and $W^V$. We assume they are for a single head, so $d_k=d_v=d$

Using $W^Q$ as an example
- First initialize an empty tensor `W_q` in the dimension of $d\times d_k$, using the `torch.empty()` function. Then initialize it with `nn.init.xavier_normal_()`.
- After `W_q` is initialized, obtain the query matrix `Q` with a multiplication between `X` and `W_q`, using `torch.matmul()`.

In [ ]:
torch.manual_seed(0) # Do not remove this line

n_heads = 1

### START YOUR CODE ###
d_k = d // n_heads # Compute d_k

W_q = None
W_k = None
W_v = None

# Compute Q, K, V
Q = None
K = None
V = None
### END YOUR CODE ###


# Test
assert Q.size() == (N, d_k)
assert K.size() == (N, d_k)
assert V.size() == (N, d_k)

print('Q.size():', Q.size())
print('Q[:,0]:', Q[:,0].data.numpy())
print('K.size():', K.size())
print('K[:,0]:', K[:,0].data.numpy())
print('V.size():', V.size())
print('V[:,0]:', V[:,0].data.numpy())

# You should expect to see the following results:
# Q.size(): torch.Size([3, 512])
# Q[:,0]: [-0.45352045 -0.40904033  0.18985942]
# K.size(): torch.Size([3, 512])
# K[:,0]: [ 1.509987   -0.5503683   0.44788954]
# V.size(): torch.Size([3, 512])
# V[:,0]: [ 0.43034226  0.00162293 -0.1317436 ]

Lastly, compute the attention scores $\alpha$ and the weighted output

Following the equation:
$$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V
$$

*Hint*:
- $\alpha = \text{softmax}(\frac{QK^\top}{\sqrt{d_k}})$, where you can use `torch.nn.functional.softmax()` to compute the softmax. Pay attention to the `dim` parameter.
- The weighted output is the multiplication between $\alpha$ and $V$. Pay attention to their dimensions: $\alpha$ is of shape $N\times N$, and $\alpha_{ij}$ is the attention score from the $i$-th to the $j$-th word. 
- The weighted output is of shape $N\times d_v$, and here we assume $d_k=d_v$.

In [ ]:
### START YOUR CODE ###
alpha = None
output = None
### END YOUR CODE ###


# Test
assert alpha.size() == (N, N)
assert output.size() == (N, d_k)

print('alpha.size():', alpha.size())
print('alpha:', alpha.data.numpy())
print('output.size():', output.size())
print('output[:,0]:', output[:,0].data.numpy())

# You should expect to see the following output:
# alpha.size(): torch.Size([3, 3])
# alpha: [[0.78344566 0.14102352 0.07553086]
#  [0.25583813 0.18030964 0.5638523 ]
#  [0.09271843 0.2767209  0.63056064]]
# output.size(): torch.Size([3, 512])
# output[:,0]: [ 0.32742795  0.03610666 -0.04272257]

## T2. Mask Future Tokens

First, create a binary mask tensor of size $N\times N$, which is lower triangular, with the diagonal and upper triangle set to 0.

*Hint*: Use `torch.tril` and `torch.ones`.

In [ ]:
### START YOUR CODE ###
mask = None
### END YOUR CODE ###

# Test
print('mask:', mask.data.numpy())

# You should expect to see the following output:
# mask: [[1. 0. 0.]
#  [1. 1. 0.]
#  [1. 1. 1.]]

Use the mask to fill the corresponding future cells in $QK^\top$ with $-\infty$ (`-np.inf`), and then pass it to softmax to compute the new attention scores.

*Hint*: Use `torch.Tensor.masked_fill` function to selectively fill the upper triangle area of the result.

In [ ]:
### START YOUR CODE ###
new_alpha = None
### END YOUR CODE ###


# Test
print('new_alpha:', new_alpha.data.numpy())

# You should expect to see the following results:
# new_alpha: [[1.         0.         0.        ]
#  [0.5865858  0.41341412 0.        ]
#  [0.09271843 0.2767209  0.63056064]]

Lastly, the output should also be updated:

In [ ]:
### START YOUR CODE ###
new_output = None
### END YOUR CODE ###

# Test
print('new_output.size():', new_output.size())
print('new_output[:,0]:', new_output[:,0].data.numpy())

# You should expect to see the following results:
# new_output.size(): torch.Size([3, 512])
# new_output[:,0]: [ 0.43034226  0.2531036  -0.04272257]

## T3. Integrate Multiple Heads

Finally, integrate the above implemented functions into the `ToyMultiHeadAttention` class.

For the convenience of later integration, we define a configuration dictionary using the hyper-parameter setep from GPT-2.

**Note**:

- In this class, the weight matrices `W_q`, `W_k`, and `W_v` are defined as tensors of size $d\times d$. Thus, the output $Q=XW^Q$ is of size $N\times d$.

- Then we reshape $Q$ (and $K$, $V$ as well) into the tensor `Q_` of shape $N\times h\times d_k$, where $h$ is the number of heads (`n_heads`) and $d_k = d // h$. Similar operations are applied to $K$ and $V$. 

- The multiplication $QK^\top$ is now between two tensors of shape $N\times h\times d_k$, `Q_` and `K_`, and the output is of size $h\times N \times N$. Thus, you need to use `torch.matmul` and `torch.permute` properly to make the dimensions of `Q_`, `K_`, and `V_` be in the correct order.

- Also, remember to apply the future mask to each attention head's output. You can use `torch.repeat` to replicate the mask for `n_heads` times.

In [ ]:
# GPT configuration
GPT_CONFIG_124M = {
    "vocab_size": 50257,    # Vocabulary size
    "context_length": 1024, # Context length
    "emb_dim": 768,         # Embedding dimension
    "n_heads": 12,          # Number of attention heads
    "n_layers": 12,         # Number of layers
    "drop_rate": 0.1,       # Dropout rate
    "qkv_bias": False       # Query-Key-Value bias
}


# Toy MultiHeadAttention
class ToyMultiHeadAttention(nn.Module):
    def __init__(self, cfg):
        super(ToyMultiHeadAttention, self).__init__()
        d_model = cfg["emb_dim"]
        n_heads = cfg["n_heads"]
        assert d_model % n_heads == 0
        self.d_k = d_model // n_heads
        self.n_heads = n_heads

        self.W_q = nn.Parameter(torch.empty(d_model, d_model))
        self.W_k = nn.Parameter(torch.empty(d_model, d_model))
        self.W_v = nn.Parameter(torch.empty(d_model, d_model))

        nn.init.xavier_normal_(self.W_q)
        nn.init.xavier_normal_(self.W_k)
        nn.init.xavier_normal_(self.W_v)
        
    def forward(self, X):
        N = X.size(0)
        
        ### START YOUR CODE ###
        Q = None
        K = None
        V = None
        Q_ = None
        K_ = None
        V_ = None

        # Raw attention scores
        alpha = None
        # Apply the mask
        mask = None
        alpha = None
        # Softmax
        alpha = None

        output = None
        ### END YOUR CODE ###

        return output

In [ ]:
# Test
torch.manual_seed(0)

MY_CONFIG = GPT_CONFIG_124M.copy()
MY_CONFIG["emb_dim"] = 512
MY_CONFIG["n_heads"] = 1

multi_head_attn = ToyMultiHeadAttention(MY_CONFIG)
output = multi_head_attn(X)

assert output.size() == (N, d)
print('output.size():', output.size())
print('output[:,0]:', output[:,0].data.numpy())

# You should expect to see the following results:
# output.size(): torch.Size([3, 512])
# output[:,0]: [ 0.43034226  0.2531036  -0.04272257]

**Note** that the above output size and values should be the same as the previous one, as we used `n_heads=1`.

## T4. FeedForward and LayerNorm

OK, next we go ahead a define other components needed in a Transformer block: `FeedForward` and `LayerNorm`. 

`FeedForward` is basically a small neural network with two linear layers and a non-linear activation in-between. A variant of ReLU, Gaussian Error Linear Unit (GELU) is used, which was the origianl option in GPT-2.

In [ ]:
# Gaussian Error Linear Unit (GELU) activation
class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) * 
            (x + 0.044715 * torch.pow(x, 3))
        ))

In [ ]:
import matplotlib.pyplot as plt

gelu, relu = GELU(), nn.ReLU()

# Some sample data
x = torch.linspace(-3, 3, 100)
y_gelu, y_relu = gelu(x), relu(x)

plt.figure(figsize=(8, 3))
for i, (y, label) in enumerate(zip([y_gelu, y_relu], ["GELU", "ReLU"]), 1):
    plt.subplot(1, 2, i)
    plt.plot(x, y)
    plt.title(f"{label} activation function")
    plt.xlabel("x")
    plt.ylabel(f"{label}(x)")
    plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# FeedForward
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)

In [ ]:
# Test FeedForward
ffn = FeedForward(GPT_CONFIG_124M)

x = torch.rand(2, 3, 768) # input shape: [batch_size, num_token, emb_size]
out = ffn(x)
print(out.shape)

Next, we implement `LayerNorm`, which centers the activations from a neural network layer around mean of 0, and normalized variance to 1.

In [ ]:
# Example
torch.manual_seed(0)

x = torch.randn(2, 5) # 2 examples with 5 dimensions each
layer = nn.Sequential(nn.Linear(5, 6), nn.ReLU())
out = layer(x)

print("Output before normalization:")
print(out)

In [ ]:
# Compute and means and vars for the outputs
means = out.mean(dim=-1, keepdim=True)
vars = out.var(dim=-1, keepdim=True)

print(f"Means: {means}")
print(f"Vars: {vars}")

In [ ]:
# Subtracting the means and dividing by the square root of vars
out_norm = (out - means) / torch.sqrt(vars + 1e-5)

print("Output after normalization:")
print(out_norm)

In [ ]:
# Examine the new means and vars
new_means = out_norm.mean(dim=-1, keepdim=True)
new_vars = out_norm.var(dim=-1, keepdim=True)

torch.set_printoptions(sci_mode=False) # disable scientific notation for readability
print(f"New means: {new_means}")
print(f"New vars: {new_vars}")

In addition to the above-demonstrated operations, `LayerNrom` introduces two additional learnable parameters `scale` and `shift`, to automatically adjust during training, i.e., learning appropriate scaling and shifting that best suit the data.

The way to add learnable parameter to the computational graph in PyTorch is to use `nn.Parameter`.

In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False) # unbiased=False, => biased estimator of variance
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift

In [ ]:
# Test LayerNorm
ln = LayerNorm(emb_dim=6)
out_ln = ln(out)

means = out_ln.mean(dim=-1, keepdim=True)
vars = out_ln.var(dim=-1, unbiased=False, keepdim=True)

print(f"Means: {means}")
print(f"Vars: {vars}")

The final piece of the jigsaw is **shortcut**, or residual connection. 

A shortcut connection creates an alternative shorter path for gradients to flow through the network, which mitigates vanishing gradient problems.

It is achieved by adding the output from an earlier layer to the output from a later layer, skipping one or more layers in between.

In [ ]:
# Toy Deep Neural Network to demonstrate shortcut connection

class ToyDeepNeuralNetwork(nn.Module):
    def __init__(self, layer_sizes, use_shortcut):
        super().__init__()
        self.use_shortcut = use_shortcut
        self.layers = nn.ModuleList([
            nn.Sequential(nn.Linear(layer_sizes[0], layer_sizes[1]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[1], layer_sizes[2]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[2], layer_sizes[3]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[3], layer_sizes[4]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[4], layer_sizes[5]), GELU())
        ])

    def forward(self, x):
        for layer in self.layers:
            layer_output = layer(x) # compute the output of current layer
            if self.use_shortcut and x.shape == layer_output.shape:
                x = x + layer_output # apply shortcut
            else:
                x = layer_output
        return x

# Helper function to print values of gradients
def print_gradients(model, x):
    output = model(x)
    target = torch.tensor([[0.]])
    # Calculate loss based on how close the target and output are
    loss = nn.MSELoss()
    loss = loss(output, target)
    loss.backward() # backprop
    # Print
    for name, param in model.named_parameters():
        if 'weight' in name:
            print(f"{name} has gradient mean of {param.grad.abs().mean().item()}")

In [ ]:
layer_sizes = [3, 3, 3, 3, 3, 1]  

sample_input = torch.tensor([[1., 0., -1.]])

torch.manual_seed(123)
model_without_shortcut = ToyDeepNeuralNetwork(
    layer_sizes, use_shortcut=False
)
print_gradients(model_without_shortcut, sample_input)

In [ ]:
torch.manual_seed(123)
model_with_shortcut = ToyDeepNeuralNetwork(
    layer_sizes, use_shortcut=True
)
print_gradients(model_with_shortcut, sample_input)

As we can see shorcut connections prevent gradients from vanishing.

---

Finally, let's combine `MultiHeadAttention`, `FeedForward`, `LayerNorm` and shortcut together to build a `TransformerBlock` module.

Here, `MultiHeadAttention` is implemented differently from the Toy MHA in order to be compatible.

In [ ]:
from utils import MultiHeadAttention

class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in = cfg["emb_dim"],
            d_out = cfg["emb_dim"],
            context_length = cfg["context_length"],
            num_heads = cfg["n_heads"], 
            dropout = cfg["drop_rate"],
            qkv_bias = cfg["qkv_bias"]
        )
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        # Shortcut connection for attention block
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)  # Shape [batch_size, num_tokens, emb_size]
        x = self.drop_shortcut(x)
        x = x + shortcut  # Add the original input back

        # Shortcut connection for feed forward block
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut  # Add the original input back

        return x

In [ ]:
torch.manual_seed(123)

x = torch.rand(2, 4, 768)  # Shape: [batch_size, num_tokens, emb_dim]
block = TransformerBlock(GPT_CONFIG_124M)
output = block(x)

print("Input shape:", x.shape)
print("Output shape:", output.shape)

## T5. Overview of a Transformer-based GPT-like Language Model

Generative Pretrained Transformer (GPT) is a decoder-only transformer-based architecture for language modeling.

Here we lay out the necessary components of building such a model.

**Notes**:
- In addition to the `TranformerBlock` we just implemented above, a language modeling head submodule `out_head` is added. It maps the embedding outputs from the final layer to logits of `vocab_size`.
- Input token IDs are converted to embeddings by `tok_emb` and added with an absolute positional embedding `pos_emb`.

In [ ]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        
        # Use a placeholder for TransformerBlock
        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        
        # Use a placeholder for LayerNorm
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(
            cfg["emb_dim"], cfg["vocab_size"], bias=False
        )

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

Let's prepare some data and test the model

In [ ]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

batch = []
txt1 = "Every effort moves you"
txt2 = "Every day holds a"

batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))
batch = torch.stack(batch, dim=0)
print(batch)

The model takes the bach of tokenized IDs and produces some logits. Watch the size of the output.

In [ ]:
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)

out = model(batch)
print("Input batch:\n", batch)
print("\nOutput shape:", out.shape)
print(out)